### Error assignment 

In this script we will calculate the values ​​of the sum rules for the LHAPDF values ​​using the number of replicas provided by the library.

In [ ]:
import lhapdf
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.integrate import quad

# PIDs of interest
pids = [21, 1, 2, 3, 4, -1, -2, -3, -4]

idx = {
    "g": 0,
    "d": 1,
    "u": 2,
    "s": 3,
    "c": 4,
    "d_bar": 5,
    "u_bar": 6,
    "s_bar": 7,
    "c_bar": 8,
}

# Load PDF set
pdfs = lhapdf.mkPDFs("NNPDF40_nnlo_as_01180")
n_replicas = len(pdfs)

# Fixed Q value
Q_FIXED = 91.19
Q2_FIXED = Q_FIXED**2

print(f"Using fixed Q = {Q_FIXED:.4f} GeV")
print(f"Using fixed Q2 = {Q2_FIXED:.4f} GeV^2")

# Integration limits
# Do not use exactly x = 0 because valence integrands divide by x.
x_min = 1e-9
x_max = 1.0


def get_xfx(pdf_member, x, q2):
    """
    Wrapper around LHAPDF.
    Returns a dictionary pid -> x*f_pid(x, Q2).
    """
    return pdf_member.xfxQ2(float(x), float(q2))


def momentum_integrand(x, pdf_member):
    """
    Integrand for total momentum:
        sum_i x f_i(x, Q2)
    """
    xfx = get_xfx(pdf_member, x, Q2_FIXED)
    return sum(xfx[pid] for pid in pids)


def valence_integrand(x, pdf_member, q_pid, qbar_pid):
    """
    Integrand for valence sum rule:
        f_q(x, Q2) - f_qbar(x, Q2)

    Since LHAPDF gives x*f(x,Q2), we compute:
        (x*f_q - x*f_qbar) / x
    """
    xfx = get_xfx(pdf_member, x, Q2_FIXED)
    return (xfx[q_pid] - xfx[qbar_pid]) / x


def integrate_quad(func, *args):
    """
    Small wrapper around scipy.integrate.quad.
    """
    value, error = quad(
        func,
        x_min,
        x_max,
        args=args,
        epsabs=1e-6,
        epsrel=1e-4,
    )
    return value, error


def get_error_with_quad(pdfs):
    all_results = []
    all_quad_errors = []

    for k, pdf_member in enumerate(tqdm(pdfs, desc="Processing replicas with quad")):
        mom, mom_err = integrate_quad(momentum_integrand, pdf_member)

        uv, uv_err = integrate_quad(
            valence_integrand,
            pdf_member,
            2,  # u
            -2,  # u_bar
        )

        dv, dv_err = integrate_quad(
            valence_integrand,
            pdf_member,
            1,  # d
            -1,  # d_bar
        )

        sv, sv_err = integrate_quad(
            valence_integrand,
            pdf_member,
            3,  # s
            -3,  # s_bar
        )

        cv, cv_err = integrate_quad(
            valence_integrand,
            pdf_member,
            4,  # c
            -4,  # c_bar
        )

        all_results.append([mom, uv, dv, sv, cv])
        all_quad_errors.append([mom_err, uv_err, dv_err, sv_err, cv_err])

    all_results = np.array(all_results)
    all_quad_errors = np.array(all_quad_errors)

    central_values = all_results[0, :]
    mean_values = np.mean(all_results[1:, :], axis=0)
    std_values = np.std(all_results[1:, :], axis=0, ddof=1)

    central_quad_errors = all_quad_errors[0, :]
    mean_quad_errors = np.mean(all_quad_errors[1:, :], axis=0)

    df = pd.DataFrame(
        {
            "Sum Rule": [
                "Total Momentum",
                "u Valence",
                "d Valence",
                "s Valence",
                "c Valence",
            ],
            "Theoretical": [1.0, 2.0, 1.0, 0.0, 0.0],
            "LHAPDF Central": central_values,
            "Replica Mean": mean_values,
            "Replica Std": std_values,
            "Mean ± Std": [
                f"{m:.6f} ± {s:.6f}" for m, s in zip(mean_values, std_values)
            ],
            "Quad Error Central": central_quad_errors,
            "Mean Quad Error": mean_quad_errors,
        }
    )

    return df


report_df = get_error_with_quad(pdfs)

print(
    report_df.round(
        {
            "Theoretical": 2,
            "LHAPDF Central": 6,
            "Replica Mean": 6,
            "Replica Std": 6,
            "Quad Error Central": 2,
            "Mean Quad Error": 2,
        }
    )
)


# Usage example
report_df = get_error_with_quad(pdfs)
print(
    report_df.round(
        {"Theoretical": 2, "LHAPDF Central": 6, "Replica Mean": 6, "Replica Std": 6}
    )
)


Using fixed Q = 91.1900 GeV
Using fixed Q2 = 8315.6161 GeV^2
LHAPDF 6.5.6 loading all 101 PDFs in set NNPDF40_nnlo_as_01180
NNPDF40_nnlo_as_01180, version 1; 101 PDF members


Processing replicas with quad:   0%|          | 0/101 [00:00<?, ?it/s]/tmp/ipykernel_25827/151401690.py:72: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  value, error = quad(
Processing replicas with quad: 100%|██████████| 101/101 [00:01<00:00, 72.92it/s]


         Sum Rule  Theoretical  LHAPDF Central  Replica Mean  Replica Std  \
0  Total Momentum          1.0        0.976604      0.976607     0.000080   
1       u Valence          2.0        2.000561      2.000379     0.001177   
2       d Valence          1.0        1.001051      1.000180     0.001199   
3       s Valence          0.0        0.000439      0.000415     0.000609   
4       c Valence          0.0       -0.000018     -0.000018     0.000016   

             Mean ± Std  Quad Error Central  Mean Quad Error  
0   0.976607 ± 0.000080                 0.0              0.0  
1   2.000379 ± 0.001177                 0.0              0.0  
2   1.000180 ± 0.001199                 0.0              0.0  
3   0.000415 ± 0.000609                 0.0              0.0  
4  -0.000018 ± 0.000016                 0.0              0.0  


Processing replicas with quad:   0%|          | 0/101 [00:00<?, ?it/s]/tmp/ipykernel_25827/151401690.py:72: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  value, error = quad(
Processing replicas with quad: 100%|██████████| 101/101 [00:01<00:00, 78.03it/s]

         Sum Rule  Theoretical  LHAPDF Central  Replica Mean  Replica Std  \
0  Total Momentum          1.0        0.976604      0.976607     0.000080   
1       u Valence          2.0        2.000561      2.000379     0.001177   
2       d Valence          1.0        1.001051      1.000180     0.001199   
3       s Valence          0.0        0.000439      0.000415     0.000609   
4       c Valence          0.0       -0.000018     -0.000018     0.000016   

             Mean ± Std  Quad Error Central  Mean Quad Error  
0   0.976607 ± 0.000080        8.034209e-05     8.048870e-05  
1   2.000379 ± 0.001177        1.164426e-04     1.050883e-04  
2   1.000180 ± 0.001199        2.294760e-05     5.156694e-05  
3   0.000415 ± 0.000609        5.034542e-07     6.391698e-07  
4  -0.000018 ± 0.000016        8.911915e-07     4.974805e-07  
